# 01 — Exploratory Data Analysis: HAM10000

> **Academic Prototype Notice:** This project is a university computer vision assignment. The model is not clinically validated and must not be used as a replacement for professional dermatological diagnosis.

This notebook covers:
1. Dataset loading and inspection
2. Missing value analysis
3. Class distribution and imbalance analysis
4. Sample images from each class
5. Discussion of visually similar classes

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

from src.dataset import prepare_dataframe
from src.config import CLASS_NAMES, CLASS_FULL_NAMES, OUTPUTS_DIR
from src.eda import plot_class_distribution, plot_sample_images

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

## 1. Load Dataset

In [ ]:
df = prepare_dataframe()
print("Shape:", df.shape)
df.head()

### Observations
The dataset contains **10,015 images** across **7 diagnostic classes**. Each row represents one dermoscopic image and includes metadata columns: `lesion_id`, `image_id`, `dx` (diagnosis label), `dx_type` (method of confirmation), `age`, `sex`, and `localization` (body site).

The `lesion_id` column is important: a single physical lesion can have **multiple images** from different angles. This means naive random splitting can leak information — the same lesion could appear in both train and test sets. We address this by splitting at the **lesion level** (group split).

## 2. Missing Value Analysis

In [ ]:
missing = df[["age", "sex", "localization"]].isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_df)

### Observations
- **`age`** has ~57 missing values (~0.57% of the dataset). These rows are retained since age is metadata only; the model does not use it as input.
- **`sex`** and **`localization`** are generally complete or have negligible missing entries.
- **Missing values do not affect model training** because we only use the image pixels and the `dx` label. However, they are worth noting for future work that uses clinical metadata as auxiliary inputs.

## 3. Class Distribution & Imbalance Analysis

In [ ]:
counts = df["dx"].value_counts().reindex(CLASS_NAMES)
pct    = (counts / counts.sum() * 100).round(1)
dist_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Full Name": [CLASS_FULL_NAMES[c] for c in CLASS_NAMES],
    "Count": counts.values,
    "% of Dataset": pct.values,
})
print(dist_df.to_string(index=False))

In [ ]:
plot_class_distribution(df, OUTPUTS_DIR / "class_distribution.png")
print("Saved → outputs/class_distribution.png")

### Class Imbalance — Why It Matters

The HAM10000 dataset is **severely imbalanced**:

| Dominant class | Minority classes |
|---|---|
| `nv` (Melanocytic Nevi) makes up ~67% of all images | `df` and `vasc` together account for less than 2% |

**Why this is a problem:**
- A naive model that always predicts `nv` would achieve ~67% accuracy while being completely useless for detecting melanoma or other rare but dangerous lesions.
- **Accuracy alone is a misleading metric** for this dataset. A model scoring 80% accuracy may still miss most melanoma cases.

**How we address it:**
1. **Weighted CrossEntropyLoss** — assigns higher loss penalty to minority class mistakes, forcing the model to pay attention to rare classes.
2. **Macro F1-score as the primary metric** — macro F1 treats every class equally regardless of size, so it will be low if any class (including rare ones) performs poorly.
3. **Stratified splitting** — ensures all 7 classes appear in train, val, and test in proportional amounts.
4. **Data augmentation** — increases effective training examples for all classes, partially compensating for imbalance.

## 4. Sample Images per Class

In [ ]:
plot_sample_images(df, OUTPUTS_DIR / "sample_images.png")
print("Saved → outputs/sample_images.png")

### Visual Observations

Inspecting sample images reveals why this is a challenging classification problem:

- **`nv` (Melanocytic Nevi)** — typically round or oval, uniform pigmentation, well-defined borders. Very common.
- **`mel` (Melanoma)** — irregular borders, asymmetric shape, uneven colour (may show black, brown, red, or white regions). **Clinically critical to detect.**
- **`bkl` (Benign Keratosis-like)** — often has a "stuck-on" warty appearance with milia-like cysts.
- **`bcc` (Basal Cell Carcinoma)** — pearly or waxy appearance, may show telangiectasia (visible blood vessels).
- **`akiec` (Actinic Keratoses)** — scaly, rough surface, often with erythema (redness).
- **`df` (Dermatofibroma)** — firm, small, often shows a central white patch.
- **`vasc` (Vascular)** — red/purple colour is distinctive, caused by blood vessel proliferation.

## 5. Visually Similar Classes — The Core Challenge

In [ ]:
# Show 3 images side-by-side for mel, nv, bkl to highlight visual similarity
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
fig.suptitle("Visually Similar Classes: mel / nv / bkl\n"
             "These three classes share overlapping colour, texture, and shape features",
             fontsize=13, fontweight="bold", y=1.01)

hard_classes = ["mel", "nv", "bkl"]
titles = {
    "mel": "Melanoma (MALIGNANT)",
    "nv":  "Melanocytic Nevi (benign)",
    "bkl": "Benign Keratosis (benign)",
}
title_colors = {"mel": "crimson", "nv": "steelblue", "bkl": "seagreen"}

for row, cls in enumerate(hard_classes):
    samples = df[df["dx"] == cls].sample(3, random_state=row)
    for col, (_, row_data) in enumerate(samples.iterrows()):
        try:
            img = Image.open(row_data["image_path"]).convert("RGB")
            axes[row][col].imshow(img)
        except Exception:
            axes[row][col].text(0.5, 0.5, "image not found", ha="center", va="center")
            axes[row][col].set_facecolor("#eee")
        if col == 0:
            axes[row][col].set_ylabel(titles[cls], fontsize=10,
                                       color=title_colors[cls], fontweight="bold")
        axes[row][col].axis("off")

plt.tight_layout()
save_path = OUTPUTS_DIR / "visually_similar_classes.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

### Why mel / nv / bkl Are Easily Confused

These three classes are the primary source of model errors in dermoscopy classification:

| Feature | mel | nv | bkl |
|---|---|---|---|
| Colour | Brown-black, may be multicolour | Brown, uniform | Brown-tan, uniform |
| Border | Irregular, asymmetric | Regular, symmetric | Variable |
| Texture | Irregular | Smooth | Rough/warty |
| Clinical risk | **HIGH — malignant** | Low | Low |

**The clinical danger:** A model that confuses `mel` with `nv` or `bkl` produces a **false negative** — a malignant lesion is classified as benign. This is far more dangerous than the reverse error (false positive), which would only lead to an unnecessary but harmless biopsy.

This is why we **specifically monitor mel recall** as a medical safety metric throughout all experiments. A recall below 0.70 for melanoma would be considered a clinically unacceptable result.

**Note:** Even experienced dermatologists achieve ~75–85% accuracy on dermoscopy; CNN models with transfer learning can approach or exceed this on benchmark sets.

## 6. Additional Metadata Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Patient Metadata Distribution", fontsize=13, fontweight="bold")

# Age distribution
df["age"].dropna().hist(bins=20, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

# Sex distribution
df["sex"].value_counts().plot(kind="bar", ax=axes[1], color=["steelblue", "salmon"], edgecolor="white")
axes[1].set_title("Sex Distribution")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

# Top localizations
df["localization"].value_counts().head(8).plot(
    kind="barh", ax=axes[2], color="mediumpurple", edgecolor="white")
axes[2].set_title("Top 8 Body Localizations")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "metadata_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/metadata_distribution.png")

### Metadata Observations
- **Age:** Most patients are between 40–70 years old. Skin cancer risk increases with age, consistent with the dataset demographics.
- **Sex:** Slightly more male patients than female in this dataset.
- **Localization:** The back and lower extremity are the most common lesion sites. This matches clinical epidemiology where sun-exposed areas are high-risk zones for skin cancer.

These metadata features are **not used as model inputs** in this project (image-only approach). Future work could explore multi-modal models combining image features with structured patient metadata.

## 7. Lesion ID Analysis — Data Leakage Risk

In [ ]:
total_images  = len(df)
total_lesions = df["lesion_id"].nunique()
multi_image   = df.groupby("lesion_id").size()
multi_pct     = (multi_image > 1).mean() * 100

print(f"Total images      : {total_images:,}")
print(f"Unique lesions    : {total_lesions:,}")
print(f"Avg images/lesion : {total_images/total_lesions:.2f}")
print(f"Lesions with >1 image: {multi_pct:.1f}%")
print()
print("Images per lesion (distribution):")
print(multi_image.value_counts().sort_index().head(10))

### Data Leakage Risk

A significant fraction of lesions have **multiple images** (different crops or angles of the same physical lesion). If we split randomly at the image level, the same lesion's images can appear in both training and test sets. This means the model may "recognise" a test image it has effectively seen before, leading to **overly optimistic test metrics**.

**Our solution:** We perform a **group split by `lesion_id`** — all images of a given lesion are kept together in the same split. This is the correct scientific approach and is noted as an advanced requirement in the project specification.

## 8. EDA Summary

| Finding | Impact on Modeling |
|---|---|
| Severe class imbalance (`nv` = 67%) | Use Weighted CrossEntropyLoss + Macro F1 |
| mel/nv/bkl are visually similar | Expect confusion between these classes; monitor mel recall |
| Multi-image lesions risk data leakage | Group split by `lesion_id` |
| ~57 missing age values | No impact (age not used as model input) |
| 10,015 total images, 7 classes | Moderate-sized dataset; transfer learning is appropriate |

These findings directly inform every subsequent design decision in this project.